# Demo 7: vicinity of thermal balance in the Jaynes-Cummings model

This notebook follows the nonclassical region around the horizontal thermal-balance line

$$
\nu_e = \frac{p_e}{1 - 2p_e},
$$

in the Jaynes-Cummings $(\tau, \nu)$ plane. Here $\nu$ is the same mean thermal photon number used in `demo_6_counting_matrix_jc.ipynb`. At $\nu_e$ the state is classical by symmetry. Rather than scanning a rectangular grid, we locate the lower and upper nonclassical bands by bisection:

- lower-side transition: between $\nu=10^{-3}$ and $\nu_e$,
- upper-side transition: between $\nu_e$ and the upper cutoff, here $\nu=20$.

The comparison uses analytical K, the ideal photoelectric photon-counting matrix criterion from Krishnaswamy et al. arXiv:2603.06237v1, and the LP solver. Majorization is intentionally disabled here.

The default $\tau$ array matches `demo_6`: positive times from `1e-3` to `10.001` with step `0.02`. A side is stored as `NaN` when its outer endpoint is classical, or when the symmetry point is unexpectedly classified as nonclassical.

In [ ]:
# Make the project importable when running from the notebooks/ directory.
import os, sys
sys.path.insert(0, os.path.abspath('..'))

from src.config import apply_thread_limits
apply_thread_limits()

import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm


In [ ]:
from src.analytical.criteria_counting_matrix import analytical_c
from src.analytical.criteria_k import analytical_k
from src.analytical.jaynes_cummings import Pjaycum
from src.config import SCAN
from src.solvers.lp_solver import solve_semi_infinite_lp
from src.utils.numerics import inverse_factorials


## Helpers

The boundary finder uses $\nu_e$ as the classical symmetry reference. It searches the lower side only when $\nu_{min}$ is nonclassical, and the upper side only when $\nu_{max}$ is nonclassical.
Each bracket is assumed to contain a single transition. Bisection does not map disconnected bands or find bands whose outer endpoint is undetected. Failed predicate evaluations also produce `NaN` and are recorded in the failure flags.


In [ ]:
def jc_probs(nu, tau, pe, NP):
    nu_eval = max(float(nu), nu_eval_floor)
    return np.array([Pjaycum(n=n, nu=nu_eval, tau=float(tau), pe=pe) for n in range(NP)], dtype=float)


def make_classifier(kind, NP, pe):
    indx = np.arange(NP, dtype=int)
    fact = inverse_factorials(int(indx.max()))
    bounds = [SCAN.lambda_bound] * len(indx)

    if kind == 'ana_k':
        def is_nonclassical(tau, nu):
            probs = jc_probs(nu, tau, pe, NP).reshape(1, 1, -1)
            nonclass, _ = analytical_k(probs, fact, indx=indx)
            return bool(nonclass[0, 0])
    elif kind == 'cm':
        def is_nonclassical(tau, nu):
            probs = jc_probs(nu, tau, pe, NP).reshape(1, 1, -1)
            nonclass, _ = analytical_c(probs, indx=indx)
            return bool(nonclass[0, 0])
    elif kind == 'lp':
        def is_nonclassical(tau, nu):
            probs = jc_probs(nu, tau, pe, NP)
            result = solve_semi_infinite_lp(
                probs,
                indx,
                fact,
                bounds,
                mu_init=(0,),
                tol=SCAN.tol,
                max_iter=SCAN.max_iter,
                imtol=SCAN.imtol,
                warn_nonphysical=False,
            )
            return bool(np.isfinite(result['num']) and result['num'] > SCAN.bias)
    else:
        raise ValueError("kind must be 'ana_k', 'cm', or 'lp'")

    return is_nonclassical


def evaluate_predicate(tau, nu, predicate, failures, *, where):
    try:
        return predicate(tau, nu)
    except Exception as exc:
        failures.append((where, float(tau), float(nu), type(exc).__name__, str(exc)))
        return None


def bisect_transition(tau, predicate, low, high, *, target_high, failures, where):
    lo = float(low)
    hi = float(high)
    for _ in range(max_bisection_iter):
        mid = 0.5 * (lo + hi)
        if hi - lo <= nu_tol:
            break
        value = evaluate_predicate(tau, mid, predicate, failures, where=where)
        if value is None:
            return np.nan
        if value == target_high:
            hi = mid
        else:
            lo = mid
    return hi


def thermal_balance_boundaries(tau_values, predicate, *, label):
    lower = np.full(len(tau_values), np.nan, dtype=float)
    upper = np.full(len(tau_values), np.nan, dtype=float)
    balance = np.zeros(len(tau_values), dtype=bool)
    outer_low = np.zeros(len(tau_values), dtype=bool)
    outer_high = np.zeros(len(tau_values), dtype=bool)
    balance_failed = np.zeros(len(tau_values), dtype=bool)
    lower_failed = np.zeros(len(tau_values), dtype=bool)
    upper_failed = np.zeros(len(tau_values), dtype=bool)
    failures = []

    for i, tau in enumerate(tqdm(tau_values, desc=label)):
        at_balance = evaluate_predicate(tau, nu_e, predicate, failures, where='balance')
        if at_balance is None:
            balance_failed[i] = True
            continue

        balance[i] = at_balance
        if at_balance:
            continue

        at_low = evaluate_predicate(tau, nu_min, predicate, failures, where='lower endpoint')
        if at_low is None:
            lower_failed[i] = True
        else:
            outer_low[i] = at_low
            if at_low:
                lower[i] = bisect_transition(
                    tau, predicate, nu_min, nu_e,
                    target_high=False, failures=failures, where='lower bisection',
                )
                lower_failed[i] = not np.isfinite(lower[i])

        at_high = evaluate_predicate(tau, nu_max, predicate, failures, where='upper endpoint')
        if at_high is None:
            upper_failed[i] = True
        else:
            outer_high[i] = at_high
            if at_high:
                upper[i] = bisect_transition(
                    tau, predicate, nu_e, nu_max,
                    target_high=True, failures=failures, where='upper bisection',
                )
                upper_failed[i] = not np.isfinite(upper[i])

    if failures:
        where, tau, nu, exc_type, message = failures[0]
        print(
            f'{label}: skipped {len(failures)} failed predicate evaluation(s); '
            f'first at {where}, tau={tau:.6g}, nu={nu:.6g}: {exc_type}: {message}'
        )

    return {
        'tau': np.asarray(tau_values, dtype=float),
        'lower': lower,
        'upper': upper,
        'balance_nonclassical': balance,
        'outer_low_nonclassical': outer_low,
        'outer_high_nonclassical': outer_high,
        'balance_failed': balance_failed,
        'lower_failed': lower_failed,
        'upper_failed': upper_failed,
    }


def boundary_summary(name, trace):
    n_lower = np.count_nonzero(np.isfinite(trace['lower']))
    n_upper = np.count_nonzero(np.isfinite(trace['upper']))
    n_balance = np.count_nonzero(trace['balance_nonclassical'])
    n_low = np.count_nonzero(trace['outer_low_nonclassical'])
    n_high = np.count_nonzero(trace['outer_high_nonclassical'])
    n_failed = sum(np.count_nonzero(trace.get(key, False)) for key in ('balance_failed', 'lower_failed', 'upper_failed'))
    print(
        f"{name}: balance {n_balance}/{len(trace['tau'])}, "
        f"low endpoint {n_low}, high endpoint {n_high}, "
        f"lower {n_lower}, upper {n_upper}, failed {n_failed}"
    )


def _archive_key(label):
    return str(label).lower().replace(' ', '_').replace('-', '_')


def save_boundary_traces(filename, traces, *, extra=None):
    os.makedirs(os.path.dirname(os.path.abspath(filename)), exist_ok=True)
    archive = {
        'pe': np.array(pe, dtype=float),
        'nu_e': np.array(nu_e, dtype=float),
        'nu_min': np.array(nu_min, dtype=float),
        'nu_max': np.array(nu_max, dtype=float),
        'nu_tol': np.array(nu_tol, dtype=float),
        'tau_values': np.asarray(tau_values, dtype=float),
        'labels': np.array(list(traces.keys()), dtype=str),
    }
    if extra:
        archive.update(extra)

    for label, trace in traces.items():
        prefix = _archive_key(label)
        for key, value in trace.items():
            archive[f'{prefix}_{key}'] = np.asarray(value)

    np.savez_compressed(filename, **archive)
    print(f'saved: {filename}')


def load_boundary_traces(filename):
    with np.load(filename) as archive:
        labels = [str(label) for label in archive['labels']]
        traces = {}
        for label in labels:
            prefix = _archive_key(label)
            tau = archive[f'{prefix}_tau'].copy()
            traces[label] = {
                'tau': tau,
                'lower': archive[f'{prefix}_lower'].copy(),
                'upper': archive[f'{prefix}_upper'].copy(),
                'balance_nonclassical': archive[f'{prefix}_balance_nonclassical'].astype(bool),
                'outer_low_nonclassical': archive[f'{prefix}_outer_low_nonclassical'].astype(bool),
                'outer_high_nonclassical': archive[f'{prefix}_outer_high_nonclassical'].astype(bool),
                'balance_failed': archive[f'{prefix}_balance_failed'].astype(bool) if f'{prefix}_balance_failed' in archive else np.zeros_like(tau, dtype=bool),
                'lower_failed': archive[f'{prefix}_lower_failed'].astype(bool) if f'{prefix}_lower_failed' in archive else np.zeros_like(tau, dtype=bool),
                'upper_failed': archive[f'{prefix}_upper_failed'].astype(bool) if f'{prefix}_upper_failed' in archive else np.zeros_like(tau, dtype=bool),
            }
        meta = {
            'pe': float(archive['pe']),
            'nu_e': float(archive['nu_e']),
            'nu_min': float(archive['nu_min']),
            'nu_max': float(archive['nu_max']),
            'nu_tol': float(archive['nu_tol']),
            'tau_values': archive['tau_values'].copy(),
        }

    return traces, meta


## Settings

The LP calls use the standard scan settings from `SCAN`. The bisection tolerance controls only how accurately the plotted boundary is localized in $\nu$.

In [ ]:
pe = 0.48
if not 0 <= pe < 0.5:
    raise ValueError('The balance line nu_e = pe / (1 - 2 pe) requires 0 <= pe < 0.5.')

nu_e = pe / (1.0 - 2.0 * pe)
nu_min = 1e-3
nu_max = 20.0
if not nu_min < nu_e < nu_max:
    raise ValueError("Choose nu_min < nu_e < nu_max to bracket both sides of thermal balance.")
nu_eval_floor = 1e-10  # Pjaycum contains divisions by nu; this represents the zero endpoint numerically.

tau_step = 0.02
tau_values = np.arange(1e-3, 10.001, tau_step)
nu_tol = 1e-10
max_bisection_iter = 40

print('pe =', pe, '| nu_e =', nu_e)
print('tau points =', len(tau_values), '| tau_step =', tau_step, '| nu_tol =', nu_tol, '| SCAN.bias =', SCAN.bias)


## Boundary comparison for $N_P=10$

This cell compares the analytical K criterion, the counting-matrix criterion, and the LP boundary using the same photon-number cutoff.

In [ ]:
NP_reference = 10
criteria = {
    'analytical K': ('ana_k', 'tab:red', '-.'),
    'counting matrix': ('cm', 'tab:green', '--'),
    'LP': ('lp', 'black', '-'),
}

boundaries_np10 = {}
for label, (kind, _, _) in criteria.items():
    predicate = make_classifier(kind, NP_reference, pe)
    trace = thermal_balance_boundaries(tau_values, predicate, label=f'{label}, NP={NP_reference}')
    boundaries_np10[label] = trace
    boundary_summary(f'{label}, NP={NP_reference}', trace)

save_boundary_traces(
    '../results/demo_7_thermal_balance_NP10_pe048.npz',
    boundaries_np10,
    extra={'NP_reference': np.array(NP_reference, dtype=int)},
)


In [ ]:
plt.figure(figsize=(7, 5))
for label, (_, color, linestyle) in criteria.items():
    trace = boundaries_np10[label]
    plt.plot(trace['tau'], trace['lower'], color=color, linestyle=linestyle, lw=2.2, label=f'{label} lower')
    plt.plot(trace['tau'], trace['upper'], color=color, linestyle=linestyle, lw=2.2, label=f'{label} upper')

plt.axhline(nu_e, color='0.45', lw=1.5, linestyle=':', label=r'$\nu_e$')
plt.xlabel(r'$\tau$')
plt.ylabel(r'$\nu$')
plt.title(rf'JC bisection boundaries, $N_P={NP_reference}$, $p_e={pe}$')
plt.xlim(0, float(np.max(tau_values)))
plt.ylim(0, min(nu_max, 8 * nu_e))
plt.legend(ncols=2, fontsize=8)
plt.tight_layout()
plt.show()


## Reloaded NP10 archive: logarithmic distance from thermal balance

The saved NP10 archive can be loaded without rerunning the bisection. The plot below shows the distance of each lower and upper boundary from the classical symmetry line, restricted to the vicinity $0.5\nu_e \le \nu \le 1.5\nu_e$. A straight trend on the semilog scale indicates exponential approach.


In [ ]:
np10_archive = '../results/demo_7_thermal_balance_NP10_pe048.npz'
loaded_np10, meta_np10 = load_boundary_traces(np10_archive)
nu_e_loaded = meta_np10['nu_e']
distance_limit = 0.5 * nu_e_loaded

plt.figure(figsize=(7, 5))
for label, (_, color, linestyle) in criteria.items():
    trace = loaded_np10[label]
    tau = trace['tau']
    lower_distance = np.abs(trace['lower'] - nu_e_loaded)
    upper_distance = np.abs(trace['upper'] - nu_e_loaded)

    lower_mask = np.isfinite(lower_distance) & (lower_distance > 0) & (lower_distance <= distance_limit)
    upper_mask = np.isfinite(upper_distance) & (upper_distance > 0) & (upper_distance <= distance_limit)

    plt.semilogy(tau[lower_mask], lower_distance[lower_mask], color=color, linestyle=linestyle, lw=2.0, label=f'{label} lower')
    plt.semilogy(tau[upper_mask], upper_distance[upper_mask], color=color, linestyle='--', lw=2.0, label=f'{label} upper')

plt.xlabel(r'$\tau$')
plt.ylabel(r'$|\nu_\mathrm{boundary} - \nu_e|$')
plt.title(rf'Distance from thermal balance, loaded $N_P={NP_reference}$ archive')
plt.xlim(0, float(np.max(meta_np10['tau_values'])))
plt.ylim(nu_tol, distance_limit)
plt.grid(True, which='both', axis='y', alpha=0.25)
plt.legend(ncols=2, fontsize=8)
plt.tight_layout()
plt.show()


## Counting-matrix comparison across photon-number cutoffs

This cell compares the counting-matrix boundaries for several values of $N_P$. The counting-matrix criterion is much faster and more robust than LP for larger cutoffs, so this comparison can use larger photon-number windows.


In [ ]:
NP_values = [10, 20, 30]

cm_boundaries_by_np = {}
for NP in NP_values:
    predicate = make_classifier('cm', NP, pe)
    trace = thermal_balance_boundaries(tau_values, predicate, label=f'CM, NP={NP}')
    cm_boundaries_by_np[NP] = trace
    boundary_summary(f'CM, NP={NP}', trace)

cm_by_np_archive = '../results/demo_7_thermal_balance_cm_by_NP_pe048.npz'
save_boundary_traces(
    cm_by_np_archive,
    {f'NP{NP}': trace for NP, trace in cm_boundaries_by_np.items()},
    extra={'NP_values': np.array(NP_values, dtype=int)},
)


In [ ]:
plt.figure(figsize=(7, 5))
colors = plt.get_cmap('tab10')(np.linspace(0, 1, len(NP_values)))

for color, NP in zip(colors, NP_values):
    trace = cm_boundaries_by_np[NP]
    plt.plot(trace['tau'], trace['lower'], color=color, lw=2.0, label=rf'$N_P={NP}$ lower')
    plt.plot(trace['tau'], trace['upper'], color=color, lw=2.0, linestyle='--', label=rf'$N_P={NP}$ upper')

plt.axhline(nu_e, color='0.45', lw=1.5, linestyle=':', label=r'$\nu_e$')
plt.xlabel(r'$\tau$')
plt.ylabel(r'$\nu$')
plt.title(rf'Counting-matrix boundaries near thermal balance, $p_e={pe}$')
plt.xlim(0, float(np.max(tau_values)))
plt.ylim(0, min(nu_max, 8 * nu_e))
plt.legend(ncols=2, fontsize=8)
plt.tight_layout()
plt.show()


## Reloaded counting-matrix cutoff archive: logarithmic distance

The saved counting-matrix cutoff archive can be loaded without rerunning the bisection. The plot below shows $|\nu_\mathrm{boundary}-\nu_e|$ on a logarithmic scale for boundaries in the vicinity $0.5\nu_e \le \nu \le 1.5\nu_e$.


In [ ]:
loaded_cm_by_np, meta_cm_by_np = load_boundary_traces(cm_by_np_archive)
nu_e_loaded = meta_cm_by_np['nu_e']
distance_limit = 0.5 * nu_e_loaded

plt.figure(figsize=(7, 5))
colors = plt.get_cmap('tab10')(np.linspace(0, 1, len(NP_values)))

for color, NP in zip(colors, NP_values):
    trace = loaded_cm_by_np[f'NP{NP}']
    tau = trace['tau']
    lower_distance = np.abs(trace['lower'] - nu_e_loaded)
    upper_distance = np.abs(trace['upper'] - nu_e_loaded)

    lower_mask = np.isfinite(lower_distance) & (lower_distance > 0) & (lower_distance <= distance_limit)
    upper_mask = np.isfinite(upper_distance) & (upper_distance > 0) & (upper_distance <= distance_limit)

    plt.semilogy(tau[lower_mask], lower_distance[lower_mask], color=color, lw=2.0, label=rf'$N_P={NP}$ lower')
    plt.semilogy(tau[upper_mask], upper_distance[upper_mask], color=color, linestyle='--', lw=2.0, label=rf'$N_P={NP}$ upper')

plt.xlabel(r'$\tau$')
plt.ylabel(r'$|\nu_\mathrm{boundary} - \nu_e|$')
plt.title(rf'Counting-matrix distance from thermal balance, $p_e={meta_cm_by_np["pe"]}$')
plt.xlim(0, float(np.max(meta_cm_by_np['tau_values'])))
plt.ylim(nu_tol, distance_limit)
plt.grid(True, which='both', axis='y', alpha=0.25)
plt.legend(ncols=2, fontsize=8)
plt.tight_layout()
plt.show()


## Further examples

- [demo_1_lp_solver](demo_1_lp_solver.ipynb): Single-point LP witnesses, a Nelder–Mead cross-check, and a Jaynes–Cummings example.
- [demo_2_analytical_vs_numerical](demo_2_analytical_vs_numerical.ipynb): Probability-plane scans comparing LP with analytical K and M criteria.
- [demo_3_jaynes_cummings](demo_3_jaynes_cummings.ipynb): Jaynes–Cummings maps with K/M comparisons and plotting of multiple boundary branches.
- [demo_4_probabilistic_boson_addition](demo_4_probabilistic_boson_addition.ipynb): Probabilistic boson addition, reversed scan direction, tighter LP tolerances, probability-scale diagnostics, and CSV save/load; the two-boson example uses optional QuTiP.
- [demo_5_counting_matrix_vs_numerics](demo_5_counting_matrix_vs_numerics.ipynb): Probability-plane comparisons of LP, K, M, and counting-matrix criteria, including nonconsecutive indices and saved-scan plotting.
- [demo_6_counting_matrix_jc](demo_6_counting_matrix_jc.ipynb): Jaynes–Cummings counting-matrix comparisons for several photon cutoffs, plus a selected-index K/M/CM comparison.
